# RNN and LSTM Quick Start

This notebook shows the minimum steps to:
1. Load a text dataset
2. Prepare it for a recurrent model
3. Train a SimpleRNN and an LSTM
4. Predict sentiment on new sentences

In [ ]:
# !pip install tensorflow

In [ ]:
import numpy as np
import tensorflow as tf
import keras

keras.utils.set_random_seed(42)
print('TensorFlow:', tf.__version__)
print('Keras     :', keras.__version__)

## 1. Load the data

Keras ships the IMDB dataset - 50,000 movie reviews already encoded as integers.
Each integer is a word ID; each label is `1` (positive) or `0` (negative).

In [ ]:
from keras.datasets import imdb
from keras.utils import pad_sequences

VOCAB_SIZE  = 10_000   # keep only the 10k most common words
MAX_LEN     = 200      # truncate / pad every review to this length

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

print(f'Train samples : {len(x_train)}')
print(f'Test  samples : {len(x_test)}')
print(f'Label classes : {np.unique(y_train)}  (0=negative, 1=positive)')

In [ ]:
# Pad so every review is the same length (shorter ones get zeros at the front)
x_train = pad_sequences(x_train, maxlen=MAX_LEN)
x_test  = pad_sequences(x_test,  maxlen=MAX_LEN)

print('x_train shape:', x_train.shape)   # (25000, 200)
print('x_test  shape:', x_test.shape)    # (25000, 200)

In [ ]:
# Use a small subset so training is fast on a laptop CPU
x_tr, y_tr = x_train[:8_000], y_train[:8_000]
x_te, y_te = x_test[:2_000],  y_test[:2_000]

print(f'Training on {len(x_tr)} reviews, testing on {len(x_te)}')

## 2. Decode a review (optional - just to see what the data looks like)

In [ ]:
word_index    = imdb.get_word_index()
index_to_word = {v + 3: k for k, v in word_index.items()}
index_to_word.update({0: '<pad>', 1: '<start>', 2: '<unk>'})

sample_review = ' '.join(index_to_word.get(i, '?') for i in x_train[0] if i != 0)
print('Label :', 'POSITIVE' if y_train[0] == 1 else 'NEGATIVE')
print('Review:', sample_review[:300], '...')

## 3. Build the models

Both models are identical except for one layer: `SimpleRNN` vs `LSTM`.

```
integers → Embedding → recurrent layer → Dense(sigmoid) → probability
```

In [ ]:
from keras import Sequential
from keras.layers import Embedding, SimpleRNN, LSTM, Dense

def build_model(recurrent_layer, name):
    keras.utils.set_random_seed(42)
    model = Sequential([
        keras.Input(shape=(MAX_LEN,)),
        Embedding(VOCAB_SIZE, 32, mask_zero=True),  # word-id → 32-dim vector
        recurrent_layer,                            # SimpleRNN or LSTM
        Dense(1, activation='sigmoid'),             # output: probability 0..1
    ], name=name)
    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

rnn_model  = build_model(SimpleRNN(32), 'SimpleRNN')
lstm_model = build_model(LSTM(32),      'LSTM')

rnn_model.summary()

## 4. Train

In [ ]:
EPOCHS     = 3
BATCH_SIZE = 64

print('Training SimpleRNN ...')
hist_rnn = rnn_model.fit(
    x_tr, y_tr,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    verbose=1,
)

In [ ]:
print('Training LSTM ...')
hist_lstm = lstm_model.fit(
    x_tr, y_tr,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    verbose=1,
)

## 5. Evaluate on the test set

In [ ]:
_, rnn_acc  = rnn_model.evaluate(x_te, y_te, verbose=0)
_, lstm_acc = lstm_model.evaluate(x_te, y_te, verbose=0)

print(f'SimpleRNN test accuracy : {rnn_acc:.3f}')
print(f'LSTM      test accuracy : {lstm_acc:.3f}')

## 6. Predict on new sentences

We encode new text the same way the IMDB dataset was encoded:
1. lowercase and split into words
2. look each word up in `word_index` (add 3 for the reserved tokens)
3. pad to `MAX_LEN`

In [ ]:
def encode(text):
    tokens = [1]  # 1 = <start>
    for word in text.lower().split():
        idx = word_index.get(word)
        tokens.append(idx + 3 if idx and idx + 3 < VOCAB_SIZE else 2)
    return tokens

def predict(model, sentences):
    encoded = pad_sequences([encode(s) for s in sentences], maxlen=MAX_LEN)
    probs   = model.predict(encoded, verbose=0)
    for sentence, p in zip(sentences, probs):
        label = 'POSITIVE' if p[0] > 0.5 else 'NEGATIVE'
        print(f'  {p[0]:.3f}  {label:8s}  {sentence}')

In [ ]:
my_reviews = [
    'this movie was absolutely wonderful i loved every minute',
    'a complete waste of time the acting was terrible and boring',
    'one of the best films i have ever seen truly brilliant',
    'i really wanted to like it but it was just dull and far too long',
]

print('--- SimpleRNN ---')
predict(rnn_model, my_reviews)

print()
print('--- LSTM ---')
predict(lstm_model, my_reviews)

In [ ]:
# Try your own sentence here
predict(lstm_model, [
    'the story was predictable but the acting saved it',
])

## Quick recap

| Step | What happened |
|---|---|
| `imdb.load_data` | integers already - no manual tokenisation needed |
| `pad_sequences` | all reviews padded to the same length |
| `Embedding` | integer → dense vector (learned during training) |
| `SimpleRNN` / `LSTM` | reads words one-by-one; final hidden state = sentence summary |
| `Dense(sigmoid)` | summary → probability (>0.5 = positive) |

LSTM usually beats SimpleRNN on longer sequences because its gates stop early words from being forgotten.

## Assignment

**Task 1 - Write your own reviews**
Add 5 sentences of your own to `my_reviews` and run `predict()`.
Include at least one sarcastic sentence like *"oh great, another superhero movie"*.
Note where the model gets it wrong.

**Task 2 - Change the hidden size**
Try `LSTM(8)` and `LSTM(64)`. Fill in this table:

| Units | Test Accuracy |
|-------|--------------|
| 8     |              |
| 32    |  (baseline)  |
| 64    |              |

What trend do you notice?

---

## Bonus - Meet GRU (Gated Recurrent Unit)

GRU is a lighter version of LSTM. Instead of 3 gates (forget / input / output), it uses only 2:

| Gate | What it does |
|------|-------------|
| **Reset gate** | how much of the past memory to forget |
| **Update gate** | how much new information to let in |

Fewer parameters → trains faster, uses less memory.
Often matches LSTM accuracy on smaller datasets.

In Keras it is a one-word swap:

```python
from keras.layers import GRU
model = build_model(GRU(32), 'GRU')
```

**Bonus task:** swap `LSTM(32)` for `GRU(32)`, retrain, and compare test accuracy and training time with LSTM.